# *

This notebook reproduces every figure and table of Sections&nbsp;4.1&ndash;4.3 and Appendix&nbsp;C.1 of the paper **end-to-end**. Tabular experiments use pure NumPy; the continuous&ndash;state linear&ndash;quadratic benchmark (Section&nbsp;4.3) uses a small PyTorch MLP (`torch.float32`, single-threaded).

| Module | Paper reference |
|---|---|
| `rmdp/duality.py`         | Prop.&nbsp;3.4: Blanchet&ndash;Murthy dual $\mathcal F^{\lambda}$ |
| `rmdp/actor_critic.py`    | Algorithm&nbsp;1 &mdash; tabular regime (Steps&nbsp;1&ndash;5) |
| `rmdp/lq_actor_critic.py` | Algorithm&nbsp;1 &mdash; scalar linear&ndash;quadratic regime |
| `rmdp/exact_dp.py`        | Robust DP benchmark (Thm.&nbsp;2.7 + Rmk.&nbsp;3.5) |
| `rmdp/kl_dp.py`           | KL-robust benchmark (Donsker&ndash;Varadhan) |
| `rmdp/lq_riccati.py`      | Kim&ndash;Yang robust Riccati (Proposition&nbsp;4.1) |
| `rmdp/lq_grid.py`         | Discrete-adversary LQ value iteration |

**Contents.** &nbsp;1.&nbsp;Coin toss &mdash; Section&nbsp;4.1 &nbsp;&nbsp; 2.&nbsp;Supply chain &mdash; Section&nbsp;4.2 &nbsp;&nbsp; 3.&nbsp;Self-exciting bandits &mdash; Appendix&nbsp;C.1 &nbsp;&nbsp; 4.&nbsp;Robust linear&ndash;quadratic control &mdash; Section&nbsp;4.3

Figures land in `../figures/` as `.pdf` (Type-42 TrueType, Type-42 TrueType) and `.png`. End-to-end wall-time on a 2024 MacBook (1 thread): $\approx$ 90&nbsp;min, of which $\approx$ 35&nbsp;min is the LQ actor&ndash;critic across four $\varepsilon$ values. Tabular sections finish in well under 10&nbsp;min and reproduce the exact-DP benchmarks bit-for-bit.

In [ ]:
# Setup.
import os, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(1)
warnings.filterwarnings('ignore', category=RuntimeWarning)
sys.path.insert(0, os.path.abspath('..'))

from rmdp import actor_critic, exact_dp, kl_dp, plotting
from rmdp import lq_grid, lq_riccati
from rmdp.lq_actor_critic import LQACConfig, LQRobustActorCritic
from rmdp.envs import bandits, coin_toss, supply_chain, lq as lq_env

plotting.use_paper_style()
FIG = os.path.abspath(os.path.join('..', 'figures'))
os.makedirs(FIG, exist_ok=True)
np.random.seed(0)

print(f'torch {torch.__version__}  |  numpy {np.__version__}')

## 1. Coin toss &mdash; Section&nbsp;4.1

State $X_t\in\{0,\dots,n\}$ counts heads in a block of $n$ Bernoulli trials; actions $a\in\{-1,0,+1\}$ (bet-down, abstain, bet-up); reference transition $X_{t+1}\sim\mathrm{Binomial}(n,p_0)$ independent of $(x,a)$; running reward $f(t,x,a,x')=a\,\mathbf 1_{\{x'>x\}}-a\,\mathbf 1_{\{x'<x\}}-|a|\,\mathbf 1_{\{x'=x\}}$, terminal $g\equiv 0$; horizon $T=10$; ground cost $c(x,y)=|x-y|$ (Wasserstein-1). Algorithm&nbsp;1 is validated against the exact Wasserstein-robust backward induction of Corollary&nbsp;2.

In [ ]:
coin_spec = coin_toss.CoinTossSpec(n=10, T=10, p0=0.5)
P0c, rc, tc, cc, mu0c = coin_toss.build(coin_spec)

eps_coin = [0.0, 0.5, 1.0, 2.0]                    # eps = 0 -> non-robust baseline
coin = {}
t0 = time.perf_counter()
for eps in eps_coin:
    V_ex, _, pi_ex = exact_dp.solve_robust_dp(
        rc, tc, P0c, cc, eps=eps, q=1, T=coin_spec.T)
    cfg = actor_critic.ACConfig(
        eps=eps, q=1, lr=0.2, n_outer=300, log_every=25, seed=0)
    theta, hist = actor_critic.run_actor_critic(
        rc, tc, P0c, cc, mu0c, cfg, T=coin_spec.T)
    coin[eps] = dict(V_exact=V_ex, pi_exact=pi_ex, theta=theta,
                     pi_learned=actor_critic.greedy_actions(theta),
                     history=hist)
    ok = np.array_equal(coin[eps]['pi_learned'], pi_ex)
    print(f'  eps = {eps:<4}  J = {hist[-1]["J"]:+.3f}   matches exact DP = {ok}')
print(f'\nCoin-toss training: {time.perf_counter() - t0:.1f}s')

**Table&nbsp;1 &mdash;** Greedy action $a^\star_0(x)=\arg\max_a\pi_0^{\theta^\star}(x,a)$ as $\varepsilon$ grows. As ambiguity increases, the betting region shrinks around $x=5$; at $\varepsilon=2$ the agent abstains everywhere, recovering the policy reported in&nbsp;Neufeld &amp; Sester&nbsp;(2023).

In [ ]:
ACTIONS = coin_toss.ACTIONS
states  = np.arange(coin_spec.n + 1)
header  = f'{"policy":>16} | ' + ' '.join(f'{x:>2}' for x in states)
print(header);  print('-' * len(header))
for eps in eps_coin:
    tag  = 'non-robust' if eps == 0 else f'eps = {eps}'
    acts = ACTIONS[coin[eps]['pi_learned'][0]]
    print(f'{tag:>16} | ' + ' '.join(f'{int(a):>+d}' for a in acts))

**Learning curves** (Appendix&nbsp;C.2 &mdash; learning curves). Each outer iteration of Algorithm&nbsp;1 runs one backward sweep of Steps&nbsp;1&ndash;5 followed by one Adam-preconditioned actor ascent step.

In [ ]:
fig, axes = plotting.new_fig(width_frac=1.0, columns=3, height=2.1)
for ax, eps in zip(axes, [0.5, 1.0, 2.0]):
    hist    = coin[eps]['history']
    J_exact = float(mu0c @ coin[eps]['V_exact'][0])
    plotting.learning_curve(ax, hist, reference=(J_exact, 'exact DP'))
    ax.set_title(rf'$\varepsilon = {eps}$')
for ax in axes[1:]:
    ax.set_ylabel('')
fig.suptitle('Coin toss \u2014 learning curves of Algorithm\u202f1', y=1.02)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'coin_training_all'))
for eps in [0.5, 1.0, 2.0]:
    hist    = coin[eps]['history']
    J_exact = float(mu0c @ coin[eps]['V_exact'][0])
    fig2, ax = plotting.new_fig(width_frac=0.55, aspect=0.72)
    plotting.learning_curve(ax, hist, reference=(J_exact, 'exact DP'))
    ax.set_title('Coin toss \u2014 learning curve ' + rf'($\varepsilon={eps}$)')
    fig2.tight_layout()
    plotting.savefig(fig2, os.path.join(FIG, f'coin_training_eps{eps}'))
    plt.close(fig2)
plt.show()

**Figure&nbsp;1 &mdash;** Cumulative profit under model misspecification. We evaluate each learned policy over $10^4$ independent rollouts while sweeping the true bias $p_{\mathrm{true}}\in[0.3,0.7]$. As $\varepsilon$ grows, the robust policy sacrifices peak profit near $p_0$ in exchange for a flatter worst-case profile across the whole range.

In [ ]:
p_true_grid = np.linspace(0.3, 0.7, 21)

def misspec_policies(p0):
    spec = coin_toss.CoinTossSpec(n=10, T=10, p0=p0)
    P0, r, t, c, mu0 = coin_toss.build(spec)
    return spec, {eps: exact_dp.solve_robust_dp(
        r, t, P0, c, eps=eps, q=1, T=spec.T)[2]
                  for eps in [0.0, 0.5, 1.0, 2.0]}

# Compute once, share between Fig. 1 line plots and the heat-map below.
misspec = {}
for p0 in [0.5, 0.6]:
    spec, policies = misspec_policies(p0)
    profits = {eps: np.array([coin_toss.simulate_profit(
                   pi, pt, spec, n_games=10_000, seed=0).mean()
                   for pt in p_true_grid])
               for eps, pi in policies.items()}
    misspec[p0] = dict(spec=spec, policies=policies, profits=profits)

fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.6)
cols = [plotting.PALETTE[k] for k in ('grey', 'blue', 'orange', 'red')]
for ax, p0 in zip(axes, [0.5, 0.6]):
    for (eps, profits), col in zip(misspec[p0]['profits'].items(), cols):
        lab = 'non-robust' if eps == 0 else rf'$\varepsilon={eps}$'
        ax.plot(p_true_grid, profits, color=col, label=lab,
                marker='o', ms=3, linewidth=1.5)
    ax.axvline(p0, color=plotting.PALETTE['grey'], linestyle=':', lw=0.9)
    ax.set_xlabel(r'true bias $p_{\mathrm{true}}$')
    ax.set_title(rf'$p_0={p0}$')
axes[0].set_ylabel('mean cumulative profit')
axes[0].legend(loc='lower center', ncol=2)
fig.suptitle('Coin toss — cumulative profit under misspecification', y=1.02)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'coin_toss_misspecification_panel'))
for p0 in [0.5, 0.6]:
    fig2, ax = plotting.new_fig(width_frac=0.55, aspect=0.72)
    for (eps, profits), col in zip(misspec[p0]['profits'].items(), cols):
        lab = 'non-robust' if eps == 0 else rf'$\varepsilon={eps}$'
        ax.plot(p_true_grid, profits, color=col, label=lab,
                marker='o', ms=3, linewidth=1.5)
    ax.axvline(p0, color=plotting.PALETTE['grey'], linestyle=':', lw=0.9)
    ax.set_xlabel(r'true bias $p_{\mathrm{true}}$')
    ax.set_ylabel('mean cumulative profit')
    ax.set_title('Coin toss — misspecification ' + rf'($p_0={p0}$)')
    ax.legend(loc='lower right')
    fig2.tight_layout()
    suffix = '' if p0 == 0.5 else f'_p0{p0}'
    plotting.savefig(fig2, os.path.join(FIG, f'coin_toss_misspecification{suffix}'))
    plt.close(fig2)
plt.show()

**Profit heat-maps &mdash;** compact two-dimensional version of Fig.&nbsp;1.

In [ ]:
for p0 in [0.5, 0.6]:
    eps_vals = list(misspec[p0]['profits'].keys())
    profit   = np.stack([misspec[p0]['profits'][e] for e in eps_vals])

    fig, ax = plotting.new_fig(width_frac=0.55, aspect=0.55)
    vmax = max(abs(profit.min()), abs(profit.max()))
    im = ax.imshow(profit, aspect='auto', origin='lower',
                   extent=[p_true_grid.min(), p_true_grid.max(),
                           -0.5, len(eps_vals) - 0.5],
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_yticks(range(len(eps_vals)),
                  [('non-robust' if e == 0 else rf'$\varepsilon={e}$')
                   for e in eps_vals])
    ax.set_xlabel(r'true bias $p_{\mathrm{true}}$')
    ax.set_title(rf'Mean cumulative profit ($p_0={p0}$)')
    ax.axvline(p0, color='k', linestyle=':', lw=0.8)
    ax.grid(False)
    cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cbar.set_label('profit')
    fig.tight_layout()
    suffix = '' if p0 == 0.5 else f'_p0{p0}'
    plotting.savefig(fig, os.path.join(FIG, f'coin_toss_profit_heatmap{suffix}'))
    plt.show()

## 2. Supply chain &mdash; Section&nbsp;4.2

Spaces $\mathcal X=\mathcal A=\{0,\dots,n\}$ (on-hand inventory and order), post-order level $\bar x=\min(n,x+a)$, reference demand $D_t\sim\mathrm{Uniform}\{0,\dots,n\}$, next inventory $X_{t+1}=\max(0,\bar x-D_t)$, one-step cost

$$\ell(x,a,d)=h\,(\bar x-d)_+ + p\,(d-\bar x)_+ + k\,\mathbf 1_{\{a>0\}}.$$

Paper parameters: $n=10$, $\varepsilon=1$, $T=1$, $h=1$, $p=3$, $k=2$.

In [ ]:
sup_spec = supply_chain.SupplyChainSpec(n=10, T=1, h=1.0, p=3.0, k=2.0)
P0s, rs, ts, cs, mu0s = supply_chain.build(sup_spec)

V_wass, _, pi_wass = exact_dp.solve_robust_dp(
    rs, ts, P0s, cs, eps=1.0, q=1, T=sup_spec.T)
V_nom,  _, pi_nom  = exact_dp.solve_robust_dp(
    rs, ts, P0s, cs, eps=0.0, q=1, T=sup_spec.T)
V_kl,   pi_kl      = kl_dp.solve_kl_robust_dp(
    rs, ts, P0s, eta=0.25, T=sup_spec.T)

cfg_s = actor_critic.ACConfig(
    eps=1.0, q=1, lr=0.15, n_outer=500, log_every=50, seed=0)
theta_s, hist_s = actor_critic.run_actor_critic(
    rs, ts, P0s, cs, mu0s, cfg_s, T=sup_spec.T)
pi_l = actor_critic.greedy_actions(theta_s)
print(f'learned  a*(x) at t = 0 = {pi_l[0]}')
print(f'exact DP a*(x) at t = 0 = {pi_wass[0]}   match = '
      f'{np.array_equal(pi_l[0], pi_wass[0])}')

**Figure&nbsp;2 &mdash;** Learned greedy ordering policy $a^\star(x)$ at $t=0$ (left); value functions across ambiguity sets &mdash; non-robust DP, KL-robust ($\eta=0.25$), Wasserstein-robust ($\varepsilon=1$) (right).

In [ ]:
states_s = np.arange(sup_spec.n + 1)

fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.6)

axes[0].step(states_s, pi_l[0], where='mid',
             color=plotting.PALETTE['blue'], linewidth=1.8,
             label='Algorithm\u202f1 (learned)')
axes[0].step(states_s, pi_wass[0], where='mid',
             color=plotting.PALETTE['grey'], linewidth=1.3, linestyle='--',
             label='exact DP')
axes[0].set_xlabel(r'inventory $x$');  axes[0].set_ylabel(r'order $a^\star(x)$')
axes[0].set_title(r'Learned greedy policy ($t=0$, $\varepsilon=1$)')
axes[0].legend(loc='upper right')

axes[1].plot(states_s, V_nom[0],  color=plotting.PALETTE['grey'],
             marker='o', ms=3, linewidth=1.4, label='non-robust')
axes[1].plot(states_s, V_kl[0],   color=plotting.PALETTE['orange'],
             marker='s', ms=3, linewidth=1.4, label=r'KL ($\eta=0.25$)')
axes[1].plot(states_s, V_wass[0], color=plotting.PALETTE['blue'],
             marker='^', ms=3, linewidth=1.6, label=r'Wasserstein ($\varepsilon=1$)')
axes[1].set_xlabel(r'inventory $x$');  axes[1].set_ylabel(r'$V_0(x)$')
axes[1].set_title('Value across ambiguity sets')
axes[1].legend(loc='lower left')
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'supply_panel'))
plt.show()

# Standalone versions matching the paper filenames.
fig, ax = plotting.new_fig(width_frac=0.5, aspect=0.72)
ax.step(states_s, pi_l[0],    where='mid',
        color=plotting.PALETTE['blue'], linewidth=1.8,
        label='Algorithm\u202f1 (learned)')
ax.step(states_s, pi_wass[0], where='mid',
        color=plotting.PALETTE['grey'], linewidth=1.3, linestyle='--',
        label='exact DP')
ax.set_xlabel(r'inventory $x$');  ax.set_ylabel(r'order $a^\star(x)$')
ax.set_title(r'Learned greedy policy ($t=0$, $\varepsilon=1$)')
ax.legend(loc='upper right')
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'supply_learned_eps1.0'))
plt.close(fig)

fig, ax = plotting.new_fig(width_frac=0.55, aspect=0.75)
ax.plot(states_s, V_nom[0],  color=plotting.PALETTE['grey'],
        marker='o', ms=3, linewidth=1.4, label='non-robust')
ax.plot(states_s, V_kl[0],   color=plotting.PALETTE['orange'],
        marker='s', ms=3, linewidth=1.4, label=r'KL ($\eta=0.25$)')
ax.plot(states_s, V_wass[0], color=plotting.PALETTE['blue'],
        marker='^', ms=3, linewidth=1.6, label=r'Wasserstein ($\varepsilon=1$)')
ax.set_xlabel(r'inventory $x$');  ax.set_ylabel(r'$V_0(x)$')
ax.set_title('Value comparison across ambiguity sets')
ax.legend(loc='lower left')
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'supply_cost_comparison'))
plt.close(fig)

In [ ]:
fig, ax = plotting.new_fig(width_frac=0.6, aspect=0.65)
plotting.learning_curve(ax, hist_s,
                        reference=(float(mu0s @ V_wass[0]), 'exact DP'))
ax.set_title('Supply chain \u2014 learning curve ' + r'($\varepsilon=1$)')
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'supply_training_eps1.0'))
plt.show()

## 3. Self-exciting multi-armed bandits &mdash; Appendix&nbsp;C.1

State $(m,b)$ &mdash; signed outcome of the last stake and last arm played; action $(k,j)$ &mdash; stake and arm. Three arms with base probabilities $(0.4,0.55,0.35)$, excitation strength $\kappa=0.2$, $K_{\max}=2$, horizon $T=5$, Wasserstein radius $\varepsilon=0.3$.

In [ ]:
spec_b = bandits.BanditSpec(T=5, K=3, K_max=2, kappa=0.2, p=(0.4, 0.55, 0.35))
P0b, rb, tb, cb, mu0b, states_b, actions_b = bandits.build(spec_b)

V_ex_b, _, pi_ex_b = exact_dp.solve_robust_dp(
    rb, tb, P0b, cb, eps=0.3, q=1, T=spec_b.T)
cfg_b = actor_critic.ACConfig(
    eps=0.3, q=1, lr=0.2, n_outer=300, log_every=30, seed=0)
theta_b, hist_b = actor_critic.run_actor_critic(
    rb, tb, P0b, cb, mu0b, cfg_b, T=spec_b.T)
pi_l_b = actor_critic.greedy_actions(theta_b)
print(f't = 0 policy match = {np.array_equal(pi_l_b[0], pi_ex_b[0])}')

**Figure&nbsp;5 &mdash;** Greedy $(j,k)$ at $t=0$ across every state $(m,b)$. Colour indicates the chosen arm index $j$; overlayed stake $k$ is printed in each cell. Both methods agree: arm&nbsp;2 is played almost everywhere; arm&nbsp;1 takes over at states where self-excitation flips the local ranking.

In [ ]:
grid_m = [m for m in range(-spec_b.K_max, spec_b.K_max + 1) if m != 0]
grid_b = list(range(spec_b.K))
s_to_i = {tuple(s): i for i, s in enumerate(states_b)}
n_m, n_b = len(grid_m), len(grid_b)

def arm_matrix(pi0):
    arm   = np.zeros((n_m, n_b), dtype=int)
    stake = np.zeros((n_m, n_b), dtype=int)
    for im, m in enumerate(grid_m):
        for ib, b in enumerate(grid_b):
            k, j = actions_b[pi0[s_to_i[(m, b)]]]
            arm[im, ib]   = j - 1
            stake[im, ib] = k
    return arm, stake

arm_l, stake_l = arm_matrix(pi_l_b[0])
arm_e, stake_e = arm_matrix(pi_ex_b[0])

fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=3.0)
panels = [(arm_e, stake_e, 'Exact DP'),
          (arm_l, stake_l, 'Algorithm\u202f1 (learned)')]
im_last = None
for k_, (ax, (arm, stake, ttl)) in enumerate(zip(axes, panels)):
    im_last = ax.imshow(
        arm, cmap='viridis', aspect='auto', origin='lower',
        extent=[-0.5, n_b - 0.5, -0.5, n_m - 0.5],
        vmin=0, vmax=spec_b.K - 1)
    for i in range(n_m):
        for j in range(n_b):
            ax.text(j, i, f'k={stake[i, j]}', ha='center', va='center',
                    color='white', fontsize=8)
    ax.set_xticks(range(n_b), [f'$b={b}$' for b in grid_b])
    ax.set_yticks(range(n_m) if k_ == 0 else [],
                  [f'$m={m}$' for m in grid_m] if k_ == 0 else [])
    ax.set_title(ttl); ax.grid(False)
fig.subplots_adjust(wspace=0.05)
cbar = fig.colorbar(im_last, ax=axes.tolist(), shrink=0.8, pad=0.02,
                    ticks=list(range(spec_b.K)))
cbar.set_label('arm index')
plotting.savefig(fig, os.path.join(FIG, 'bandits_policies_eps0.3'))
plt.show()

In [ ]:
fig, ax = plotting.new_fig(width_frac=0.6, aspect=0.65)
plotting.learning_curve(ax, hist_b,
                        reference=(float(mu0b @ V_ex_b[0]), 'exact DP'))
ax.set_title('Self-exciting bandits \u2014 learning curve ' + r'($\varepsilon=0.3$)')
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'bandits_training_eps0.3'))
plt.show()

## 4. Robust linear&ndash;quadratic control &mdash; Section&nbsp;4.3

Dynamics $X_{t+1}=AX_t+Bu_t+\Xi w_t$ with empirical reference noise $\nu=\tfrac1N\sum_i\delta_{\hat w^{(i)}}$, running cost $x^\top Qx+u^\top Ru$, terminal $x^\top P_T x$. The discrete Wasserstein adversary redistributes mass among the $N$ atoms under $\mathcal W_q(\gamma,\nu)\le\varepsilon$.

* **Figure&nbsp;3** &mdash; Kim&ndash;Yang Riccati closed form (Proposition&nbsp;4.1).
* **Figure&nbsp;4 (top)** &mdash; exact discrete-adversary backward induction.
* **Figure&nbsp;4 (bottom)** &mdash; Algorithm&nbsp;1 on the same instance.

In [ ]:
# Figure 3 -- Kim-Yang Riccati closed form (scalar LQ).
# Stability requires lambda > max_t ||Xi^T Pi_{t+1} Xi||_2.  For the toy
# instance Xi = 1, P_T = 2 this gives lambda >= 3.
inst1d = lq_env.toy_1d_instance(seed=0)
xs = np.linspace(-3.0, 3.0, 121).reshape(-1, 1)

lam_list = [3.0, 5.0, 10.0, 30.0, 200.0]
fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.7)
colors = plotting.eps_colors(lam_list)
for lam, col in zip(lam_list, colors):
    sol = lq_riccati.robust_riccati(inst1d.to_riccati(), lam=lam)
    Pi0, r0, z0 = sol['Pi'][0], sol['r'][0], sol['z'][0]
    V_cost = np.einsum('ni,ij,nj->n', xs, Pi0, xs) + 2 * xs @ r0 + z0
    u_x = (xs @ sol['K'][0].T + sol['L'][0]).reshape(-1)
    axes[0].plot(xs.reshape(-1), -V_cost, color=col, label=rf'$\lambda={lam:g}$')
    axes[1].plot(xs.reshape(-1),  u_x,    color=col, label=rf'$\lambda={lam:g}$')
axes[0].set_xlabel(r'$x$');  axes[0].set_ylabel(r'$V_0(x)$')
axes[0].set_title('Value function')
axes[1].set_xlabel(r'$x$');  axes[1].set_ylabel(r'$u_0^\star(x)$')
axes[1].set_title('Optimal policy')
axes[0].legend(loc='lower center', ncol=2, fontsize=8)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'lq_riccati'))
plt.show()

In [ ]:
# Figure 4 (top) -- exact discrete-adversary backward induction.
eps_lq = [0.05, 0.2, 0.4, 0.7]

t0 = time.perf_counter()
dro = {eps: lq_grid.discrete_adversary_value(inst1d, eps=eps, q=2.0)
       for eps in eps_lq}
print(f'discrete-adversary grid: {time.perf_counter() - t0:.1f}s')

x_grid = dro[eps_lq[0]]['x_grid']
fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.7)
colors = plotting.eps_colors(eps_lq)
for eps, col in zip(eps_lq, colors):
    axes[0].plot(x_grid, dro[eps]['V0'], color=col, label=rf'$\varepsilon={eps}$')
    axes[1].plot(x_grid, dro[eps]['u0'], color=col, label=rf'$\varepsilon={eps}$')
axes[0].set_xlabel(r'$x$');  axes[0].set_ylabel(r'$V_0(x)$')
axes[0].set_title('Value (discrete adversary)')
axes[1].set_xlabel(r'$x$');  axes[1].set_ylabel(r'$u_0^\star(x)$')
axes[1].set_title('Optimal policy (discrete adversary)')
axes[0].legend(loc='upper right', ncol=2, fontsize=8)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'lq_discrete'))
plt.show()

In [ ]:
# Figure 4 (bottom) -- Algorithm 1 against the discrete-adversary benchmark.
# Compact  configuration: ~3 min per epsilon on a single thread.
# The paper trains for longer; the configuration here suffices for the
# qualitative match against the discrete-adversary value/policy.
learned = {}
t0 = time.perf_counter()
for eps in eps_lq:
    cfg = LQACConfig(eps=eps, q=2.0, L=3.0, seed=0,
                     n_outer=10,  log_every=5,
                     n_inner_sweeps=1, n_critic_steps=4,
                     batch_size=16, n_samples=2, n_lam=10,
                     actor_hidden=32, critic_hidden=16,
                     actor_lr=2e-2, critic_lr=1e-2)
    agent = LQRobustActorCritic(inst1d, cfg)
    logs  = agent.train(verbose=False)
    ev    = agent.evaluate(x_grid)
    learned[eps] = dict(logs=logs, eval=ev)
    print(f'  eps = {eps:<5}  J_net = {logs["J_net"][-1]:+.3f}   '
          f'J_mc = {logs["J_mc"][-1]:+.3f}   sigma = {ev["sigma"]:.3f}   '
          f'J_disc = {dro[eps]["J"]:+.3f}')
print(f'\nAlgorithm 1 training: {time.perf_counter() - t0:.1f}s')

fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.7)
for eps, col in zip(eps_lq, colors):
    axes[0].plot(x_grid, learned[eps]['eval']['V'], color=col,
                 label=rf'$\varepsilon={eps}$', linewidth=1.6)
    axes[0].plot(x_grid, dro[eps]['V0'], color=col, linestyle=':',
                 alpha=0.6, linewidth=1.2)
    axes[1].plot(x_grid, learned[eps]['eval']['u'], color=col,
                 label=rf'$\varepsilon={eps}$', linewidth=1.6)
    axes[1].plot(x_grid, dro[eps]['u0'], color=col, linestyle=':',
                 alpha=0.6, linewidth=1.2)
axes[0].set_xlabel(r'$x$');  axes[0].set_ylabel(r'$V_0(x)$')
axes[0].set_title('Algorithm 1 (solid) vs discrete adversary (dotted)')
axes[1].set_xlabel(r'$x$');  axes[1].set_ylabel(r'$u_0^\star(x)$')
axes[1].set_title('Learned mean policy')
axes[1].legend(loc='upper right', ncol=2, fontsize=8)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'lq_learned'))
plt.show()

In [ ]:
# Learning curves -- lq_training.pdf.
fig, axes = plotting.new_fig(width_frac=1.0, columns=2, height=2.7)
for eps, col in zip(eps_lq, colors):
    log = learned[eps]['logs']
    axes[0].plot(log['iter'], log['J_net'], color=col,
                 label=rf'$\varepsilon={eps}$', linewidth=1.6)
    axes[0].axhline(dro[eps]['J'], color=col, linestyle=':',
                    linewidth=0.9, alpha=0.7)
    axes[1].plot(log['iter'], log['sigma'], color=col,
                 label=rf'$\varepsilon={eps}$', linewidth=1.6)
axes[0].set_xlabel('outer iteration')
axes[0].set_ylabel(r'$\widehat J(\theta)$')
axes[0].set_title('Robust value (dotted = discrete adversary)')
axes[1].set_xlabel('outer iteration')
axes[1].set_ylabel(r'$\sigma_{\theta}$')
axes[1].set_title('Policy standard deviation')
axes[0].legend(loc='lower right', ncol=2, fontsize=8)
fig.tight_layout()
plotting.savefig(fig, os.path.join(FIG, 'lq_training'))
plt.show()

---

All figures are written to `../figures/` as `.pdf` (Type-42 TrueType fonts, Type-42 TrueType) and `.png`. To re-execute in place:

```bash
jupyter nbconvert --to notebook --execute notebooks/experiments.ipynb --inplace
```